In [0]:
from pyspark.sql.functions import col, when, upper, lower, trim, regexp_replace, to_date, current_timestamp, coalesce, lit, concat, split, size
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Ingesting unstandarized raw operationsal input rows
data = [
    (1, "  rahul sharma  ", "Mumbai", 45000, "Engineering", "1990-03-15"),
    (2, "Priya Nair", "Bangalore", 62000, "analytics", "1998-07-22"),
    (3, "Arjun mehta", "delhi", 38000, "ENINEERING", "1992-11-03"),
    (4, "Sneha Reddy", "Bangalore", 234343, None, "1985-04-22"),
    (5, "Vikram Singh", "chennai", 44440, "Analytics", "1990-05-30")
]

# Declaring a strict meta data type for schema contract
schema = StructType ([
    StructField("emp_id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("salary", StringType(), True),
    StructField("department", StringType(), True),
    StructField("date_of_birth", StringType(), True)
])

df_raw = spark.createDataFrame(data, schema)
print("Raw bronze data frame target schema mapping:")
df_raw.printSchema()
df_raw.show()

In [0]:
df_clean = df_raw \
    .withColumn("name", trim(col("name"))) \
    .withColumn("name_cleaned",
                concat(
                    upper(split(trim(col("name")), " ")[0].substr(1, 1)),
                    lower(split(trim(col("name")), " ")[0].substr(2, 100)),
                    lit(" "),
                    upper(split(trim(col("name")), " ")[1].substr(1, 1)),
                    lower(split(trim(col("name")), " ")[1].substr(2, 100))
                    )
            ) \
    .withColumn("city_standarised",
                when(lower(col("city")) == "bangalore", "Bangalore")
                .when(lower(col("city")) == "mumbai", "Mumbai")
                .when(lower(col("city")) == "chennai", "Chennai")
                .when(lower(col("city")) == "delhi", "Delhi")
                .otherwise(col("city"))
                ) \
    .withColumn("department_standardised",
                when(col("department").isNull(), "Unknown")
                .otherwise(
                    concat(upper(col("department").substr(1,1)), lower(col("department").substr(2,100)))
                        )
                ) \
    .withColumn("salary_band",
                when(col("salary") < 40000, "Junior")
                .when(col("salary") < 60000, "Mid")
                .when(col("salary") < 80000, "Senior")
                .otherwise("Principal")
                ) \
    .withColumn("loaded_at", current_timestamp())

#Triggering a Spark Action to compile and show outputs cleanly without charactrer trimming
print("Standardized Silver Record Table Layout:")
df_clean.show(truncate=False)